# RecursiveGCD (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/RecursiveGCD_cpp.ipynb)

Chapter 14 — Recursion. Runnable notebook of `RecursiveGCD`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/RecursiveGCD_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/RecursiveGCD_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Names.** There are no arrays in this chapter: variables and parameters have the same names as in the pseudo-code. The stack and the counters are global variables, and the trace is switched on with `trace = true`.

## Greatest common divisor: RecursiveGCD

**Input:** two natural numbers $n \ge 1$ and $m \ge 0$. **Output:** $\gcd(n, m)$. Tail recursion: each call makes at most one, the tree is a path. At most $2\lfloor\log_2 m\rfloor + 3$ calls, $O(\log_2 m)$ steps.

Pseudo-code: section 14.6 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [1]:
%%writefile recursive_gcd.cpp
#include <cstdarg>
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];      // cnt[r] = number of executions of line r (numbering of the lecture notes)
long long calls = 0;    // number of calls = nodes of the tree of recursive calls
int maxDepth = 0;       // depth of the recursion = maximum height of the stack
long long mults = 0;    // multiplications (FastPower)
long long moves = 0;    // moves (Hanoi)
int ev = 0;             // counter of the events of the trace
int stk[1000];          // the call stack: the arguments of the active calls (at most 1000)
int sp = 0;             // number of active calls
long long seen[64];     // seen[x] = number of calls with argument x
bool trace = false;     // true: print the trace of the execution

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
    for (int k = 0; k < 64; k++) seen[k] = 0;
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0; sp = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Line | Executions\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Total steps S = %lld\n", Total());
}

string Str(const char* fmt, ...)       // like sprintf, but returns a string
{
    char buf[256];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return string(buf);
}

void Push(int x)                       // a call starts: its frame goes on top of the stack
{
    calls++;
    stk[sp++] = x;
    if (sp > maxDepth) maxDepth = sp;
    if (x >= 0 && x < 64) seen[x]++;
}

void Pop()                             // the call returns: its frame leaves the stack
{
    sp--;
}

string StackStr()                      // "[4, 3, 2]"
{
    string s = "[";
    for (int k = 0; k < sp; k++)
    {
        if (k > 0) s += ", ";
        s += to_string(stk[k]);
    }
    return s + "]";
}

void Event(const string& text)         // numbered line of the trace, with the stack
{
    ev++;
    printf("%2d. %s   stack: %s\n", ev, text.c_str(), StackStr().c_str());
}

void Say(const string& call, const string& action)   // line of the trace: the call, indented one level for every active call
{
    printf("%-28s| %s\n", (string(2 * (sp - 1), ' ') + call).c_str(), action.c_str());
}

int FloorLog2(int m)                               // floor(log2 m) with a loop (only for the bound on the calls)
{
    int k = 0;
    while (m > 1)
    {
        m = m / 2;
        k = k + 1;
    }
    return k;
}

// RecursiveGCD(n, m)
int RecursiveGCD(int n, int m)
{
    Push(m); cnt[2]++;
    if (m == 0)
    {
        cnt[3]++;
        if (trace) Say(Str("RecursiveGCD(%d, %d)", n, m), Str("base case, returns %d", n));
        Pop();
        return n;
    }
    if (trace) Say(Str("RecursiveGCD(%d, %d)", n, m), Str("%d mod %d = %d: calls RecursiveGCD(%d, %d)", n, m, n % m, m, n % m));
    cnt[5]++;
    int g = RecursiveGCD(m, n % m);
    if (trace) Say(Str("RecursiveGCD(%d, %d)", n, m), Str("returns %d", g));
    Pop();
    return g;
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Examples of the lecture notes: RecursiveGCD(48, 18) and RecursiveGCD(252, 105), trees of calls (paths)
    int pairs[][2] = {{48, 18}, {252, 105}};
    for (auto& p : pairs)
    {
        Reset();
        trace = true;
        printf("%-28s| %s\n", "call", "action");
        int g = RecursiveGCD(p[0], p[1]);
        trace = false;
        printf("gcd(%d, %d) = %d, calls = %lld, S = %lld\n", p[0], p[1], g, calls, Total());
        printf("\n");
    }

    // Number of calls (each call executes 2 lines, hence S = 2 calls) and bound 2 floor(log2 m) + 3
    printf("      n |       m |  gcd |    calls |   S | 2 floor(log2 m) + 3\n");
    int more[][2] = {{48, 18}, {252, 105}, {89, 55}, {144, 89}, {1000, 7}, {1000000, 999999}};
    for (auto& p : more)
    {
        Reset();
        int g = RecursiveGCD(p[0], p[1]);
        printf("%7d | %7d | %4d | %8lld | %3lld | %d\n", p[0], p[1], g, calls, Total(), 2 * FloorLog2(p[1]) + 3);
    }
    return 0;
}
#endif

Writing recursive_gcd.cpp


In [2]:
!g++ -std=c++17 -O2 -o recursive_gcd recursive_gcd.cpp && ./recursive_gcd

call                        | action
RecursiveGCD(48, 18)        | 48 mod 18 = 12: calls RecursiveGCD(18, 12)
  RecursiveGCD(18, 12)      | 18 mod 12 = 6: calls RecursiveGCD(12, 6)
    RecursiveGCD(12, 6)     | 12 mod 6 = 0: calls RecursiveGCD(6, 0)
      RecursiveGCD(6, 0)    | base case, returns 6
    RecursiveGCD(12, 6)     | returns 6
  RecursiveGCD(18, 12)      | returns 6
RecursiveGCD(48, 18)        | returns 6
gcd(48, 18) = 6, calls = 4, S = 8

call                        | action
RecursiveGCD(252, 105)      | 252 mod 105 = 42: calls RecursiveGCD(105, 42)
  RecursiveGCD(105, 42)     | 105 mod 42 = 21: calls RecursiveGCD(42, 21)
    RecursiveGCD(42, 21)    | 42 mod 21 = 0: calls RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | base case, returns 21
    RecursiveGCD(42, 21)    | returns 21
  RecursiveGCD(105, 42)     | returns 21
RecursiveGCD(252, 105)      | returns 21
gcd(252, 105) = 21, calls = 4, S = 8

      n |       m |  gcd |    calls |   S | 2 floor(log2 m) + 3
     48 |

**Try it yourself.** Change the two numbers; the worst case is two consecutive Fibonacci numbers.

In [3]:
%%writefile prova_recursive_gcd.cpp
#define SOLO_LIBRERIA   // do not run the main of recursive_gcd.cpp
#include "recursive_gcd.cpp"

int main()
{
    int n = 1071, m = 462;        // <-- change the two numbers
    Reset();
    trace = true;
    printf("%-28s| %s\n", "call", "action");
    int g = RecursiveGCD(n, m);
    trace = false;
    printf("gcd(%d, %d) = %d, calls = %lld\n", n, m, g, calls);
    return 0;
}

Writing prova_recursive_gcd.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_recursive_gcd prova_recursive_gcd.cpp && ./prova_recursive_gcd

call                        | action
RecursiveGCD(1071, 462)     | 1071 mod 462 = 147: calls RecursiveGCD(462, 147)
  RecursiveGCD(462, 147)    | 462 mod 147 = 21: calls RecursiveGCD(147, 21)
    RecursiveGCD(147, 21)   | 147 mod 21 = 0: calls RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | base case, returns 21
    RecursiveGCD(147, 21)   | returns 21
  RecursiveGCD(462, 147)    | returns 21
RecursiveGCD(1071, 462)     | returns 21
gcd(1071, 462) = 21, calls = 4


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).